In [14]:
import pandas as pd
from openai import OpenAI
import os
import gradio as gr
from pathlib import Path
import sys
ROOT_DIR = Path.cwd().parent

# API setup
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("API_KEY")
)

MODEL = "openai/gpt-oss-120b"

# Load dataset
df = pd.read_csv(os.path.join(ROOT_DIR,"data","raw","german_credit_data.csv"))


def create_prompt(row):
    return f"""
You are explaining credit risk to a normal person with no finance knowledge.

Customer details:
Age: {row['Age']}
Job: {row['Job']}
Housing: {row['Housing']}
Savings: {row['Saving accounts']}
Checking balance: {row['Checking account']}
Loan amount: {row['Credit amount']}
Duration: {row['Duration']}
Purpose: {row['Purpose']}

Risk: {row['Risk']}

Explain in very simple and clear language why this person is considered {row['Risk']}.
- Use easy words (like talking to a beginner).
- No technical terms, no tables, no bullet points.
- Keep it short (3–5 lines).
- If the risk is bad, give 1–2 simple suggestions to improve.
"""


def analyze(index):
    try:
        row = df.iloc[int(index)]
        prompt = create_prompt(row)

        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are an expert financial analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"


# Minimal UI
gr.Interface(
    fn=analyze,
    inputs="text",
    outputs="text",
    title="Credit Risk Explainer (Enter Row Index)"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
